# Model Predictions (coupler_NCap_cap_matrix)

## Configuration

In [1]:
# The parameter file is where the hyperparameters are set. 
# It's reccomended to look at that file first, its interesting and you can set stuff there

from parameters import *

## Library

In [2]:
# Disable some console warnings
import os
os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices'
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import tensorflow as tf# Disable some console warnings so you can be free of them printing. 
# Comment the next two lines if you are a professional and like looking at warnings.
os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices'
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import os, gc
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from tensorflow.keras.models import load_model

## Dataset

### Load

In [5]:
def load_scaled_split(kind, split):
    # kind: 'one_hot' or 'linear'
    return np.load(f"{DATA_DIR}/npy/{split}_{kind}_encoding_scaled.npy", allow_pickle=True)

if 'Try Both' in ENCODING_TYPE:
    # one-hot
    X_train_one_hot_encoding = load_scaled_split('one_hot', 'x_train')
    X_val_one_hot_encoding   = load_scaled_split('one_hot', 'x_val')
    X_test_one_hot_encoding  = load_scaled_split('one_hot', 'x_test')

    y_train_one_hot_encoding = load_scaled_split('one_hot', 'y_train')
    y_val_one_hot_encoding   = load_scaled_split('one_hot', 'y_val')
    y_test_one_hot_encoding  = load_scaled_split('one_hot', 'y_test')

    # linear
    X_train_linear_encoding = load_scaled_split('linear', 'x_train')
    X_val_linear_encoding   = load_scaled_split('linear', 'x_val')
    X_test_linear_encoding  = load_scaled_split('linear', 'x_test')

    y_train_linear_encoding = load_scaled_split('linear', 'y_train')
    y_val_linear_encoding   = load_scaled_split('linear', 'y_val')
    y_test_linear_encoding  = load_scaled_split('linear', 'y_test')

else:
    if 'one hot' in ENCODING_TYPE:
        kind = 'one_hot'
    elif 'Linear' in ENCODING_TYPE:
        kind = 'linear'
    else:
        raise ValueError(f"Unknown ENCODING_TYPE: {ENCODING_TYPE}")

    X_train = load_scaled_split(kind, 'x_train')
    X_val   = load_scaled_split(kind, 'x_val')
    X_test  = load_scaled_split(kind, 'x_test')

    y_train = load_scaled_split(kind, 'y_train')
    y_val   = load_scaled_split(kind, 'y_val')
    y_test  = load_scaled_split(kind, 'y_test')


### Visualize

In [6]:
# Decide which model file & test set to use
chosen_path = "model/model_predict_coupler_NCap_cap_matrix.keras"      
X_test_cur = np.asarray(X_test)
y_test_cur = np.asarray(y_test)
y_encoding_format_name = encoding    

# Load y headers for labeling columns
y_headers_csv = f'y_characteristics_{y_encoding_format_name}_encoding.csv'
with open(y_headers_csv, 'r') as f:
    headers = f.readline().strip().split(',')

In [7]:
#run on CPU
tf.keras.backend.clear_session()
gc.collect()
try:
    tf.config.experimental.reset_memory_stats('GPU:0')
except Exception:
    pass

with tf.device('/CPU:0'):
    chosen_model = load_model(chosen_path, compile=False)  #dont compile it because we just need to predict
    y_pred = chosen_model.predict(X_test_cur, verbose=0)

print(f"\n—— {os.path.basename(chosen_path)} ——")
chosen_model.summary()
print(f"Samples: {len(X_test_cur)} | Targets dim: {y_test_cur.shape[1]}")

I0000 00:00:1769698498.678219    1951 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 38431 MB memory:  -> device: 0, name: NVIDIA A100 80GB PCIe MIG 4g.40gb, pci bus id: 0000:06:00.0, compute capability: 8.0
I0000 00:00:1769698499.247586    3220 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.



—— model_predict_coupler_NCap_cap_matrix.keras ——


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ fc0 (Dense)                     │ (None, 264)            │         1,848 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_relu0 (LeakyReLU)         │ (None, 264)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout0 (Dropout)              │ (None, 264)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (None, 964)            │       255,460 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_relu1 (LeakyReLU)         │ (None, 964)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout1 (Dropout)              │ (None, 964)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc2 (Dense)                     │ (None, 764)            │       737,260 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_relu2 (LeakyReLU)         │ (None, 764)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout2 (Dropout)              │ (None, 764)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc3 (Dense)                     │ (None, 664)            │       507,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_relu3 (LeakyReLU)         │ (None, 664)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout3 (Dropout)              │ (None, 664)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc4 (Dense)                     │ (None, 164)            │       109,060 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_relu4 (LeakyReLU)         │ (None, 164)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout4 (Dropout)              │ (None, 164)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 4)              │           660 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,612,248 (6.15 MB)

 Trainable params: 1,612,248 (6.15 MB)

 Non-trainable params: 0 (0.00 B)

Samples: 65 | Targets dim: 4


# Scaled

In [12]:
#use a smaller view if you want
N_SAMPLES_TO_SHOW = 50


n_samples = min(N_SAMPLES_TO_SHOW, len(X_test_cur))
n_params  = y_test_cur.shape[1]

# scaled errors
sq_errors  = (y_test_cur - y_pred) ** 2
abs_errors = np.abs(y_test_cur - y_pred)

# scaled dataframe
rows = []
for i in range(n_samples):
    top_to_top, top_to_bottom, top_to_ground, bottom_to_bottom, bottom_to_ground, ground_to_ground = X_test_cur[i, 0], X_test_cur[i, 1], X_test_cur[i, 2],X_test_cur[i, 3],X_test_cur[i, 4],X_test_cur[i, 5]
    for j in range(n_params):
        rows.append({
            "sample_idx": i,
            "top_to_bottom": top_to_bottom,
            "top_to_ground": top_to_ground,
            "bottom_to_bottom": bottom_to_bottom,
            "bottom_to_ground": bottom_to_ground,
            "ground_to_ground": ground_to_ground,
            "param": headers[j],
            "ref":  y_test_cur[i, j],
            "pred": y_pred[i, j],
            "abs_error": abs_errors[i, j],
            "sq_error":  sq_errors[i, j],
        })
df = pd.DataFrame(rows)

# save scaled predictions
out_csv = Path(f"predictions_and_errors_{y_encoding_format_name}.csv")
df.to_csv(out_csv, index=False, float_format="%.6g")
print(f"\nSaved CSV -> {out_csv.resolve()}\n")

#pretty print per-sample (scaled)
for i in range(n_samples):
    sub = df[df["sample_idx"] == i].copy()
    sub = sub[["param", "ref", "pred", "abs_error", "sq_error"]]
    header_line = (
        f"— Sample {i} — "
        f"X: top_to_top={X_test_cur[i,0]:.9g}, top_to_bottom={X_test_cur[i,1]:.9g}, top_to_ground={X_test_cur[i,2]:.9g}, bottom_to_bottom={X_test_cur[i,3]:.9g},bottom_to_ground={X_test_cur[i,4]:.9g},ground_to_ground={X_test_cur[i,5]:.9g}"
    )
    print(header_line)
    print(sub.to_string(index=False))
    print()

#print global stats (scaled)
print("Global scaled error stats:")
print("  min abs_error:", float(abs_errors.min()))
print("  median abs_error:", float(np.median(abs_errors)))
print("  max abs_error:", float(abs_errors.max()))
print("\nHere onehot/linear encoding and the MLP which maps categorical data to 1s and 0s is probably throwing off the global average. These will be rounded in the future and will probably always  round to the right number to reconstruct the correct category-- but for now it might throw off  the overall average error. In the future we might want to just have it consider the non-categorical data when finding an overall average and reporting that number.\n")


Saved CSV -> /home/olivias/ML_qubit_design/model_predict_coupler_NCap_cap_matrix/predictions_and_errors_one_hot.csv

— Sample 0 — X: top_to_top=0.00832192455, top_to_bottom=0.00565011737, top_to_ground=0.0365676091, bottom_to_bottom=0.00692052511,bottom_to_ground=0.00850165266,ground_to_ground=0.0110262114
                       param  ref      pred  abs_error  sq_error
      design_options.cap_gap  0.0  0.298992   0.298992  0.089396
    design_options.cap_width  0.2  0.085816   0.114184  0.013038
design_options.finger_length  0.0 -0.017753   0.017753  0.000315
 design_options.finger_count  0.0  0.002491   0.002491  0.000006

— Sample 1 — X: top_to_top=0.0914761888, top_to_bottom=0.0642074415, top_to_ground=0.158027636, bottom_to_bottom=0.111459797,bottom_to_ground=0.142610781,ground_to_ground=0.146250336
                       param      ref     pred  abs_error     sq_error
      design_options.cap_gap 1.000000 0.949002   0.050998 2.600751e-03
    design_options.cap_width 0.600000 0.

# Unscaled

In [13]:
#load X feature names for the X scalers
with open('X_names', 'r') as f:
    X_index_names = f.read().splitlines()

#unscale X
X_test_unscaled = np.asarray(X_test_cur.copy())
for i in range(X_test_unscaled.shape[0]):
    for j in range(X_test_unscaled.shape[1]):
        scaler = joblib.load(f'scalers/scaler_X_{X_index_names[j]}.save')
        X_test_unscaled[i, j] = scaler.inverse_transform([[X_test_unscaled[i, j]]])[0][0]

#unscale y (refs and preds)
y_test_unscaled = np.asarray(y_test_cur.copy())
y_pred_unscaled = np.asarray(y_pred.copy())
for i in range(y_test_unscaled.shape[0]):
    for j in range(y_test_unscaled.shape[1]):
        scaler = joblib.load(f'scalers/scaler_y_{headers[j]}_{y_encoding_format_name}_encoding.save')
        y_test_unscaled[i, j] = scaler.inverse_transform([[y_test_unscaled[i, j]]])[0][0]
        y_pred_unscaled[i, j] = scaler.inverse_transform([[y_pred_unscaled[i, j]]])[0][0]

# Errors (unscaled)
sq_errors_unscaled  = (y_test_unscaled - y_pred_unscaled) ** 2
abs_errors_unscaled = np.abs(y_test_unscaled - y_pred_unscaled)

#build dataframe (unscaled)
rows_unscaled = []
for i in range(min(N_SAMPLES_TO_SHOW, len(X_test_unscaled))):
    top_to_top, top_to_bottom, top_to_ground, bottom_to_bottom, bottom_to_ground, ground_to_ground = X_test_unscaled[i, 0], X_test_unscaled[i, 1], X_test_unscaled[i, 2], X_test_unscaled[i, 3], X_test_unscaled[i, 4], X_test_unscaled[i, 5]
    for j in range(n_params):
        rows_unscaled.append({
            "sample_idx": i,
            "top_to_top": top_to_top,
            "top_to_bottom": top_to_bottom,
            "top_to_ground": top_to_ground,
            "bottom_to_bottom": bottom_to_bottom,
            "bottom_to_ground": bottom_to_ground,
            "ground_to_ground": ground_to_ground,
            "param": headers[j],
            "ref_unscaled":  y_test_unscaled[i, j],
            "pred_unscaled": y_pred_unscaled[i, j],
            "abs_error_unscaled": abs_errors_unscaled[i, j],
            "sq_error_unscaled":  sq_errors_unscaled[i, j],
        })
df_unscaled = pd.DataFrame(rows_unscaled)

# save (unscaled)
out_csv_unscaled = Path(f"predictions_and_errors_unscaled_{y_encoding_format_name}.csv")
df_unscaled.to_csv(out_csv_unscaled, index=False, float_format="%.6g")
print(f"\nSaved CSV -> {out_csv_unscaled.resolve()}\n")

# Pretty print per-sample (unscaled)
for i in range(min(N_SAMPLES_TO_SHOW, len(X_test_unscaled))):
    sub = df_unscaled[df_unscaled["sample_idx"] == i].copy()
    sub = sub[["param", "ref_unscaled", "pred_unscaled", "abs_error_unscaled", "sq_error_unscaled"]]
    header_line = (
        f"— Sample {i} (Unscaled) — "
        f"X: top_to_top={X_test_unscaled[i,0]:.9g}, top_to_bottom={X_test_unscaled[i,1]:.9g}, top_to_ground={X_test_unscaled[i,2]:.9g}, bottom_to_bottom={X_test_unscaled[i,3]:.9g},bottom_to_ground={X_test_unscaled[i,4]:.9g},ground_to_ground={X_test_unscaled[i,5]:.9g}"
    )
    print(header_line)
    print(sub.to_string(index=False))
    print()

#print global stats (unscaled)
print("Global unscaled error stats:")
print("  min abs_error:", float(abs_errors_unscaled.min()))
print("  median abs_error:", float(np.median(abs_errors_unscaled)))
print("  max abs_error:", float(abs_errors_unscaled.max()))
print("\nHere onehot/linear encoding and the MLP which maps categorical data to 1s and 0s is probably throwing off the global average. These will be rounded in the future and will probably always  round to the right number to reconstruct the correct category-- but for now it might throw off  the overall average error. In the future we might want to just have it consider the non-categorical data when finding an overall average and reporting that number.\n")



Saved CSV -> /home/olivias/ML_qubit_design/model_predict_coupler_NCap_cap_matrix/predictions_and_errors_unscaled_one_hot.csv

— Sample 0 (Unscaled) — X: top_to_top=14.8635827, top_to_bottom=0.668175255, top_to_ground=13.854644, bottom_to_bottom=13.62255,bottom_to_ground=12.7645607,ground_to_ground=58.0137
                       param  ref_unscaled  pred_unscaled  abs_error_unscaled  sq_error_unscaled
      design_options.cap_gap      0.000002       0.000003        5.979847e-07       3.575856e-13
    design_options.cap_width      0.000007       0.000006        1.141844e-06       1.303809e-12
design_options.finger_length      0.000016       0.000015        5.326045e-07       2.836676e-13
 design_options.finger_count      1.000000       1.022422        2.242184e-02       5.027388e-04

— Sample 1 (Unscaled) — X: top_to_top=21.5988461, top_to_bottom=3.83709971, top_to_ground=17.3630789, bottom_to_bottom=27.51404,bottom_to_ground=23.2719519,ground_to_ground=73.35986
                       p

In [11]:
m = tf.keras.models.load_model(chosen_path, compile=True)  # loads optimizer too (if saved)

# ---- neurons per layer (your original logic, slightly generalized) ----
neurons_per_layer = [
    l.units for l in m.layers
    if isinstance(l, tf.keras.layers.Dense) and l.name.startswith("fc")
]
print("Best NEURONS_PER_LAYER:", neurons_per_layer)

# ---- dropout rate(s) from Dropout layers ----
dropouts = [
    (l.name, float(l.rate)) for l in m.layers
    if isinstance(l, tf.keras.layers.Dropout)
]
if dropouts:
    print("Best DROPOUT rate(s):", dropouts)
else:
    print("Best DROPOUT rate(s): none found")

# ---- learning rate / schedule details ----
opt = getattr(m, "optimizer", None)
if opt is None:
    print("Best learning rate: (no optimizer found on loaded model)")
else:
    lr_obj = opt.learning_rate  # can be float/Variable OR a schedule

    # Case 1: schedule (e.g., ExponentialDecay)
    if isinstance(lr_obj, tf.keras.optimizers.schedules.LearningRateSchedule):
        print("Best learning rate: (schedule)", type(lr_obj).__name__)

        # Try to print schedule config nicely
        try:
            cfg = lr_obj.get_config()
            print("LR schedule config:")
            for k, v in sorted(cfg.items()):
                print(f"  {k}: {v}")
        except Exception as e:
            print("Could not read LR schedule config:", repr(e))

        # Also print the current LR value at step=optimizer.iterations
        try:
            step = int(tf.keras.backend.get_value(opt.iterations))
            current_lr = float(lr_obj(step).numpy())
            print(f"Current LR at step {step}:", current_lr)
        except Exception as e:
            print("Could not compute current LR from schedule:", repr(e))

    # Case 2: plain numeric / variable LR
    else:
        try:
            lr_val = float(tf.keras.backend.get_value(lr_obj))
            print("Best learning rate:", lr_val)
        except Exception as e:
            print("Best learning rate: (unreadable)", repr(e))


Best NEURONS_PER_LAYER: [264, 964, 764, 664, 164]
Best DROPOUT rate(s): [('dropout0', 0.0), ('dropout1', 0.0), ('dropout2', 0.0), ('dropout3', 0.0), ('dropout4', 0.0)]
Best learning rate: 0.0007026341045275331
